# Fetching more data to help improve the accuracy of the model

We already walked through the steps before, so we will quickly repeat them below  
Resources: https://www.kaggle.com/datasets/therohk/million-headlines?resource=download

In [2]:
import pandas as pd

In [3]:
df_abc = pd.read_csv("data/news/original/abcnews-date-text.csv")

In [4]:
df_abc.head()

,publish_date,headline_text
0,20030219,aba decides against community broadcasting lic...
1,20030219,act fire witnesses must be aware of defamation
2,20030219,a g calls for infrastructure protection summit
3,20030219,air nz staff in aust strike for pay rise
4,20030219,air nz strike to affect australian travellers


Filtering the dataset

In [ ]:
df_abc.dropna(inplace=True)
df_abc.reset_index(drop=True, inplace=True)

df_abc["date"] = pd.to_datetime(df_abc["publish_date"].astype(str), format="%Y%m%d", errors="coerce")
df_abc.dropna(inplace=True) # remove rows where date conversion failed
df_abc.reset_index(drop=True, inplace=True)

min_date_abc = df_abc["date"].min()
max_date_abc = df_abc["date"].max()

df_abc["summary"] = (
    df_abc["headline_text"].fillna("").astype(str).str.strip()
).str.strip()

df_abc["source"] = "abcnews"

df_abc[["date", "summary", "source"]].to_csv(
    "data/news/filtered/abcnews_filtered.csv", index=False
)



In [ ]:
df_cnbc_filtered = pd.read_csv(
    "data/news/filtered/cnbc_filtered.csv", parse_dates=["date"]
)
df_guardian_filtered = pd.read_csv(
    "data/news/filtered/guardian_filtered.csv", parse_dates=["date"]
)
df_reuters_filtered = pd.read_csv(
    "data/news/filtered/reuters_filtered.csv", parse_dates=["date"]
)
df_abcnews_filtered = pd.read_csv(
    "data/news/filtered/abcnews_filtered.csv", parse_dates=["date"]
)

df_all = pd.concat(
    [df_cnbc_filtered, df_guardian_filtered, df_reuters_filtered, df_abcnews_filtered], ignore_index=True
)
df_all.sort_values("date", inplace=True)

,date,summary,source
53428,2003-02-19,more than 40 pc of young men drink alcohol at,abcnews
53389,2003-02-19,gold coast to hear about bilby project,abcnews
53390,2003-02-19,golf club feeling smoking ban impact,abcnews
53391,2003-02-19,govt is to blame for ethanols unpopularity opp,abcnews
53392,2003-02-19,greens offer police station alternative,abcnews


In [23]:
df_all.to_csv("data/news/filtered/all_filtered_added_data.csv", index=False)
df_all.head()

,date,summary,source
53428,2003-02-19,more than 40 pc of young men drink alcohol at,abcnews
53389,2003-02-19,gold coast to hear about bilby project,abcnews
53390,2003-02-19,golf club feeling smoking ban impact,abcnews
53391,2003-02-19,govt is to blame for ethanols unpopularity opp,abcnews
53392,2003-02-19,greens offer police station alternative,abcnews


In [8]:
%%capture
%pip install nltk

import pandas as pd
import numpy as np
import re

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

nltk.download("stopwords")
nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("wordnet")  # for lemmatization

In [9]:
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

finance_whitelist = {
    "up",
    "down",
    "above",
    "below",
    "under",
    "over",
    "rise",
    "fall",
    "not",
    "no",
    "nor",
    "neither",
    "never",
    "none",
    "more",
    "most",
    "few",
    "less",
}

stop_words.difference_update(finance_whitelist)

In [10]:
def clean_text(text):
    # 1. Lowercase all text
    text = text.lower()
    # 2. Remove punctuation
    text = re.sub(r"[^\w\s]", "", text)
    # 3. Tokenize using Punkt
    tokens = word_tokenize(text)
    # 4. Remove stopwords
    tokens = [word for word in tokens if word not in stop_words]
    # 5. Lemmatization
    tokens = [lemmatizer.lemmatize(word) for word in tokens]
    return " ".join(tokens)

In [45]:
df_news = pd.read_csv("data/news/filtered/all_filtered_added_data.csv")

df_news["clean_summary"] = df_news["summary"].apply(clean_text)

# finding all the times that are past 4pm or 16 in military time and increasing the date by 1 day unless it is friday
df_news["adj_date"] = df_news["date"]
df_news["adj_date"] = pd.to_datetime(df_news["adj_date"], format="ISO8601", errors="coerce")
df_news["date"] = pd.to_datetime(df_news["date"], format="ISO8601", errors="coerce")

# Saturday posts get moved +2 days (to Monday)
df_news.loc[df_news["adj_date"].dt.weekday == 5, "adj_date"] += pd.Timedelta(days=2)

# Sunday posts get moved +1 day (to Monday)
df_news.loc[df_news["adj_date"].dt.weekday == 6, "adj_date"] += pd.Timedelta(days=1)


df_news["adj_date"] = df_news["adj_date"].dt.normalize()
df_news = df_news.groupby("adj_date")["summary"].apply(" ".join).reset_index()
df_news.head()

,adj_date,summary
0,2003-02-19,more than 40 pc of young men drink alcohol at ...
1,2003-02-20,police tax office to fight chop chop trade pea...
2,2003-02-21,planning for intersection revamp begins plan m...
3,2003-02-24,rain brings relief for act farmers protesters ...
4,2003-02-25,qld govt reviewing cruise terminal plan qantas...


In [46]:
df_news.to_csv("data/news/cleaned/all_clean_added_data.csv", index=False)

In [49]:
# news data all sources combined
news_data = pd.read_csv("data/news/cleaned/all_clean_added_data.csv")
news_data['date'] = pd.to_datetime(news_data['adj_date'], errors='coerce')
news_data = news_data.sort_values(by="date", ascending=True).reset_index(drop=True)


# finance data

import yfinance as yf
# this is a sunday that is why there is no finicial data on it
start_date = "2003-02-19" # 2003-02-19 extra dateset
end_date = "2021-12-31" # 2021-12-31 extra dataset

raw_data = yf.download(tickers="^GSPC", start=start_date, end=end_date, interval="1d")

# flattening multi-level columns
raw_data.columns = [
    "_".join([str(c) for c in col if c != ""]) if isinstance(col, tuple) else col
    for col in raw_data.columns
]
# creating date a column from index
yfinance_data = raw_data.reset_index().rename(columns={"Date": "date"})

yfinance_data['date'] = pd.to_datetime(yfinance_data['date'], errors='coerce')

df_stock = yfinance_data.sort_values('date').reset_index(drop=True)

df_stock["pct_change"] = df_stock["Close_^GSPC"].pct_change()

# Shifting tomorrows close
df_stock["Close_T+1"] = df_stock["Close_^GSPC"].shift(-1)

# predict whether tomorrow will go UP
df_stock["Target"] = (df_stock["Close_T+1"] > df_stock["Close_^GSPC"]).astype(int)

# selecting relevant columns for our testing and training and merging into new dataframe
df_stock_refined = df_stock[["Volume_^GSPC", "pct_change", "Target", "date"]].copy()

# rename columns to lowercase because im a jerk
df_stock_refined.columns = ["volume", "pct_change", "target", "date"]

# removing the first row with NaN pct_change
# also remove the last row with NaN Target
df_stock_refined = df_stock_refined.dropna().reset_index(drop=True)

# saving refined dataframe
save_path = "data/stock/yfinance_data_cleaned.csv"
df_stock_refined.to_csv(save_path, index=False)


merged_data = pd.merge(news_data, df_stock_refined, on='date', how='outer')
if "adj_date" in merged_data.columns:
    merged_data = merged_data.drop(columns=["adj_date"])

merged_data.head()

C:\Users\Wiltj\AppData\Local\Temp\ipykernel_21604\2525220411.py:14: FutureWarning: YF.download() has changed argument auto_adjust default to True
  raw_data = yf.download(tickers="^GSPC", start=start_date, end=end_date, interval="1d")
[*********************100%***********************]  1 of 1 completed


,summary,date,volume,pct_change,target
0,more than 40 pc of young men drink alcohol at ...,2003-02-19,NaN,NaN,NaN
1,police tax office to fight chop chop trade pea...,2003-02-20,1.194100e+09,-0.009502,1.0
2,planning for intersection revamp begins plan m...,2003-02-21,1.398200e+09,0.013224,0.0
3,rain brings relief for act farmers protesters ...,2003-02-24,1.229200e+09,-0.018381,1.0
4,qld govt reviewing cruise terminal plan qantas...,2003-02-25,1.483700e+09,0.007194,0.0


In [53]:
merged_data = merged_data[merged_data['summary'].notnull()].reset_index(drop=True)


# 1. Create a "grouper" column
# We identify valid trading days. If volume exists, we keep the date.
# If volume is NaN, we set it to NaT (Not a Time) so we can fill it later.
merged_data["trading_day_group"] = np.where(
    merged_data["volume"].notnull(), merged_data["date"], pd.NaT
)

# 2. Backfill the dates
# take the date of the next valid row and pulls it UP into the previous NaN rows
merged_data["trading_day_group"] = merged_data["trading_day_group"].bfill()

# 3. Group by this new column and aggregate
# We combine the summaries and keep the financial data from the valid trading day (the last entry in the group)
shifted_data = (
    merged_data.groupby("trading_day_group")
    .agg(
        {
            "date": "last",  # Keep the actual trading date
            "summary": " ".join,  # Join the news strings together with a space
            "volume": "last",  # Take the volume from the valid day
            "pct_change": "last",  # Take the change from the valid day
            "target": "last",  # Take the target from the valid day
        }
    )
    .reset_index(drop=True)
)

# Remove any rows that remained NaN
shifted_data = shifted_data.dropna(subset=["volume"])


In [55]:
shifted_data['pct_change_lag1'] = shifted_data['pct_change'].shift(1)
shifted_data['volume_lag1'] = shifted_data['volume'].shift(1)
shifted_data['volume_5d_avg'] = shifted_data['volume'].rolling(window=5).mean()

shifted_data.to_csv('data/merged_data_additional.csv', index=False)
shifted_data.sort_values(by="date", ascending=True).reset_index(drop=True)
shifted_data.head()

,date,summary,volume,pct_change,target,pct_change_lag1,volume_lag1,volume_5d_avg
0,2003-02-20,more than 40 pc of young men drink alcohol at ...,1.194100e+09,-0.009502,1.0,NaN,NaN,NaN
1,2003-02-21,planning for intersection revamp begins plan m...,1.398200e+09,0.013224,0.0,-0.009502,1.194100e+09,NaN
2,2003-02-24,rain brings relief for act farmers protesters ...,1.229200e+09,-0.018381,1.0,0.013224,1.398200e+09,NaN
3,2003-02-25,qld govt reviewing cruise terminal plan qantas...,1.483700e+09,0.007194,0.0,-0.018381,1.229200e+09,NaN
4,2003-02-26,police believe man trapped in washed away vehi...,1.374400e+09,-0.013141,1.0,0.007194,1.483700e+09,1.335920e+09
